# Сравнение детекторов аномалий траектории

Наблюдение — точка `(time, lat, lon)`. Задача бинарная: `anomaly` против `move`. Точки `stand` в выборку не входят. В `CalculateMetrics` метка `0` — аномалия, метка `1` — `move`.

ARIMA, GP, GP-EKF, LSTM, TCN, DONUT, Anomaly Transformer, STGVAD и One-Class SVM подгоняются по точкам `move`. HMM учится на отрезках обоих классов, Isolation Forest — по всей обучающей выборке. DBSCAN и LOF считают плотность размечаемого ряда. Порог или гиперпараметр (`eps`, `nu`, `v_max`) выбирается по обучающей выборке целиком. Главный критерий порога — коэффициент Мэттьюса (MCC): он одновременно штрафует и пропуск аномалий, и ложные тревоги на штатном ходу. Среди порогов в полосе `MCC_BAND` от лучшего MCC берётся меньший distance loss, чтобы длина маршрута не разъехалась без нужды. Метрики на тесте считаются по последним 100 000 точкам. Для PR-AUC в `calculate_all_metrics` передаётся anomaly score: большее значение — более аномальная точка.

Сравнение моделей на тесте идёт так же. Сначала MCC, при близких значениях — меньший distance loss, затем point-wise F1. Абсолютное положение в состоянии фильтра есть у Калмана, фильтра частиц и GP-EKF: проекция от медианы обучающего `move`, старт с первой точки своего ряда. GP внутри GP-EKF смотрит на скорость и шаг времени, не на координаты. У остальных детекторов признаки — шаг, скорость или форма окна, поэтому район карты сам по себе score не задаёт.



In [ ]:
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.special import logsumexp
from scipy.stats import chi2
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.svm import OneClassSVM
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

SEED = 7
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cpu")
torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))

START_IND = 200_000
# В файле 419 453 строки,
# С raw-индекса 315_185 идёт сплошная anomaly.
# тест и карта последних 100_000 точек содержат оба класса.
END_IND = 350_000
TEST_SIZE = 100_000
PLOT_TAIL = 50_000
# Среди порогов с MCC не хуже лучшего больше чем на эту величину берётся меньший distance loss.
MCC_BAND = 0.02
# Окно карты: [юг, запад, север, восток]. None — автоматический масштаб по точкам.
MAP_BBOX = [50.524974, 44.809115, 54.250411, 52.029355]
# Подложка OSM (нужен интернет). False — только трек на осях.
MAP_USE_BASEMAP = True
EARTH_RADIUS_M = 6_371_000.0
N_EPOCHS = 6
N_WINDOWS = 8_000
N_PARTICLES = 32


def find_root() -> Path:
    """Корень репозитория: рядом лежат data/2.csv и пакет app."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "2.csv").exists() and (candidate / "app").exists():
            return candidate
    raise FileNotFoundError("Не найден корень проекта с data/2.csv")


path_root = find_root()
path = path_root / "data" / "2.csv"
sys.path.insert(0, str(path_root))

from app.metrics.calculate_metrics import CalculateMetrics
from app.metrics.plot_sample_track import plot_sample_track

print(f"Корень: {path_root}")
print(f"Файл: {path}")
print(f"PyTorch {torch.__version__}, потоков {torch.get_num_threads()}")



# Данные

Из `data/2.csv` берётся фрагмент `raw.iloc[START_IND:END_IND]`: `START_IND = 200_000`, `END_IND = 350_000`.


In [ ]:
raw_full = pd.read_csv(path)
raw = raw_full.iloc[START_IND:END_IND].reset_index(drop=True)

frame = raw.drop(columns=["sat", "is_water"]).copy()
frame["lat"] = pd.to_numeric(frame["lat"], errors="coerce")
frame["lon"] = pd.to_numeric(frame["lon"], errors="coerce")
keep = frame["lat"].notna() & frame["lon"].notna() & (frame["lat"] != 0.0) & (frame["lon"] != 0.0)
frame = frame.loc[keep].reset_index(drop=True)
status = frame["status"].astype(str)
n_stand = int((status == "stand").sum())
n_other = int((~status.isin(["move", "anomaly", "stand"])).sum())
frame = frame.loc[status.isin(["move", "anomaly"])].reset_index(drop=True)
order = np.argsort(pd.to_datetime(frame["time"]).to_numpy(), kind="mergesort")
frame = frame.iloc[order].reset_index(drop=True)

if len(frame) <= TEST_SIZE:
    raise RuntimeError(
        f"После очистки осталось {len(frame)} строк, этого мало для теста из {TEST_SIZE}."
    )

test_frame = frame.iloc[-TEST_SIZE:].reset_index(drop=True)
train_frame = frame.iloc[:-TEST_SIZE].reset_index(drop=True)


def nfmt(value):
    return f"{int(value):,}".replace(",", " ")


def make_pack(data):
    """Собирает одну выборку: время в секундах и бинарная метка anomaly / move."""
    stamp = pd.to_datetime(data["time"])
    status_values = data["status"].astype(str).to_numpy()
    anomaly = status_values == "anomaly"
    move = status_values == "move"
    if not np.all(anomaly | move):
        raise ValueError("В выборке остались точки не из классов move и anomaly.")
    return {
        "time": stamp.to_numpy(),
        "t": stamp.to_numpy().astype("datetime64[ns]").astype(np.int64) / 1e9,
        "lat": data["lat"].to_numpy(dtype=float),
        "lon": data["lon"].to_numpy(dtype=float),
        "anom": anomaly,
        "valid": move,
    }


def describe(name, pack):
    print(
        f"{name:<20} {nfmt(len(pack['anom'])):>10} точек   "
        f"аномалий {pack['anom'].mean():6.1%}   move {pack['valid'].mean():6.1%}"
    )


fit = make_pack(train_frame)
test = make_pack(test_frame)
if np.any(np.diff(fit["t"]) <= 0.0) or np.any(np.diff(test["t"]) <= 0.0):
    raise RuntimeError("Время в обучающей или тестовой выборке не строго возрастает.")
print(
    f"Файл целиком: {nfmt(len(raw_full))} строк, "
    f"фрагмент [{START_IND}:{END_IND}): {nfmt(len(raw))} строк"
)
print(
    f"Отброшено stand: {nfmt(n_stand)}. "
    f"Отброшено других меток: {nfmt(n_other)}. "
    f"Осталось move и anomaly: {nfmt(len(frame))}."
)
print("Признаки моделей: time, lat, lon. Классы: anomaly и move.")
describe("Обучение", fit)
describe("Тест", test)
if not np.any(test["valid"]):
    print(
        "В тесте нет точек move. Эталонная длина маршрута на тесте тогда равна нулю, "
        "а distance loss — длине пути, который модель оставила классом move."
    )


# Признаки, порог и вывод метрик

Скорость, ускорение и поворот считаются по соседним точкам уже отфильтрованного ряда: `stand` между ними нет. Форма окна — отношение длины пути к смещению за 6 шагов: у прямого хода она около 1, у дрожания на месте больше. Проекция в метрах от медианы обучающего `move` нужна фильтру Калмана, фильтру частиц и GP-EKF: их состояние стартует с первой точки своего ряда. DBSCAN, Isolation Forest и One-Class SVM смотрят на шаг, скорость и форму окна, а не на место на карте. Гауссовский процесс описывает восточный и северный шаг. GP-EKF учит остаток постоянной скорости по скорости и шагу времени.

Порог `tau` перебирается по квантилям score на всей обучающей выборке. Для каждого порога считаются MCC и distance loss. Побеждает максимальный MCC. Пороги, которые хуже не больше чем на `MCC_BAND`, сравниваются по distance loss: так срезается лишняя длина, если классы уже разделены почти одинаково. На тесте пара `(prediction, ground truth)` собирается как `(время, долгота, широта, метка)` и передаётся в `CalculateMetrics.calculate_all_metrics`. Метка `0` — аномалия. Непрерывный score, если он есть у метода, идёт в PR-AUC.

После метрик каждой модели `plot_sample_track` рисует две карты по последним 50 000 точкам теста: слева ground truth, справа разметка модели. Масштаб задаётся `MAP_BBOX` (`[юг, запад, север, восток]`); трек рисуется полностью, но обрезается по границам окна.



In [ ]:
from app.metrics.calculate_metrics import format_metric_value


def wrap_angle(angle):
    return (angle + np.pi) % (2.0 * np.pi) - np.pi


def haversine_m(lat1, lon1, lat2, lon2):
    """Геодезическое расстояние на сфере, в метрах."""
    lat1 = np.radians(np.asarray(lat1, dtype=float))
    lat2 = np.radians(np.asarray(lat2, dtype=float))
    dlat = lat2 - lat1
    dlon = np.radians(np.asarray(lon2, dtype=float) - np.asarray(lon1, dtype=float))
    dlon = (dlon + np.pi) % (2.0 * np.pi) - np.pi
    step = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * EARTH_RADIUS_M * np.arcsin(np.sqrt(np.clip(step, 0.0, 1.0)))


def bearing(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)
    dlon = np.radians(lon2 - lon1)
    east = np.sin(dlon) * np.cos(lat2)
    north = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return np.arctan2(east, north)


def project_m(lat, lon, lat0, lon0):
    east = np.radians(lon - lon0) * np.cos(np.radians(lat0)) * EARTH_RADIUS_M
    north = np.radians(lat - lat0) * EARTH_RADIUS_M
    return np.column_stack([east, north])


def motion_features(lat, lon, t_sec):
    """Скорость, ускорение, поворот курса и шаг в локальных метрах. У первой точки нули."""
    count = len(lat)
    dt = np.ones(count)
    dist = np.zeros(count)
    speed = np.zeros(count)
    acc = np.zeros(count)
    dpsi = np.zeros(count)
    dx = np.zeros(count)
    dy = np.zeros(count)
    if count < 2:
        return dt, dist, speed, acc, dpsi, dx, dy
    dt[1:] = np.diff(t_sec)
    dist[1:] = haversine_m(lat[:-1], lon[:-1], lat[1:], lon[1:])
    speed[1:] = dist[1:] / np.clip(dt[1:], 1.0, None)
    acc[1:] = np.diff(speed) / np.clip(dt[1:], 1.0, None)
    heading = np.zeros(count)
    heading[1:] = bearing(lat[:-1], lon[:-1], lat[1:], lon[1:])
    if count > 2:
        dpsi[2:] = wrap_angle(np.diff(heading[1:]))
    mean_lat = (lat[1:] + lat[:-1]) / 2.0
    dx[1:] = np.radians(lon[1:] - lon[:-1]) * np.cos(np.radians(mean_lat)) * EARTH_RADIUS_M
    dy[1:] = np.radians(lat[1:] - lat[:-1]) * EARTH_RADIUS_M
    return dt, dist, speed, acc, dpsi, dx, dy


def track_shape(lat, lon, dist, window=6):
    """Отношение длины пути к смещению за окно. Прямой ход около 1, дрожание на месте больше."""
    count = len(dist)
    shape = np.ones(count)
    if count <= window:
        return shape
    cumulative = np.cumsum(dist)
    path = cumulative[window:] - cumulative[:-window]
    span = haversine_m(lat[:-window], lon[:-window], lat[window:], lon[window:])
    shape[window:] = path / np.maximum(span, 1.0)
    return shape


def add_motion(pack):
    dt, dist, speed, acc, dpsi, dx, dy = motion_features(pack["lat"], pack["lon"], pack["t"])
    pack.update(
        dt=dt,
        dist=dist,
        speed=speed,
        acc=acc,
        dpsi=dpsi,
        dx=dx,
        dy=dy,
        shape=track_shape(pack["lat"], pack["lon"], dist),
    )
    return pack


for pack in (fit, test):
    add_motion(pack)

origin_lat = float(np.median(fit["lat"][fit["valid"]]))
origin_lon = float(np.median(fit["lon"][fit["valid"]]))
for pack in (fit, test):
    pack["xy"] = project_m(pack["lat"], pack["lon"], origin_lat, origin_lon)

normal_obs = np.column_stack([fit["speed"][fit["valid"]], fit["acc"][fit["valid"]], fit["dpsi"][fit["valid"]]])
obs_median = np.median(normal_obs, axis=0)
obs_mad = np.median(np.abs(normal_obs - obs_median), axis=0)
obs_mad = np.where(obs_mad < 1e-6, 1.0, obs_mad) * 1.4826


def kinematic_matrix(pack, sl=None):
    """(скорость, ускорение, поворот), нормированные по точкам move обучения."""
    raw_obs = np.column_stack([pack["speed"], pack["acc"], pack["dpsi"]])
    if sl is not None:
        raw_obs = raw_obs[sl]
    return np.clip((raw_obs - obs_median) / obs_mad, -25.0, 25.0)


def robust_params(matrix):
    center = np.median(matrix, axis=0)
    scale = np.subtract(*np.percentile(matrix, [75, 25], axis=0))
    scale = np.where(scale < 1e-6, 1.0, scale)
    return center, scale


def apply_robust(matrix, center, scale):
    return (matrix - center) / scale


def local_motion_matrix(pack):
    """Шаг, скорость и форма окна. Абсолютные координаты сюда не входят."""
    return np.column_stack(
        [
            pack["dx"],
            pack["dy"],
            pack["speed"],
            pack["acc"],
            pack["dpsi"],
            np.log(np.maximum(pack["shape"], 1.0)),
        ]
    )


def anomaly_f1(is_anomaly, predicted):
    true_positive = np.count_nonzero(is_anomaly & predicted)
    false_positive = np.count_nonzero(~is_anomaly & predicted)
    false_negative = np.count_nonzero(is_anomaly & ~predicted)
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
    if precision + recall == 0.0:
        return 0.0
    return 2.0 * precision * recall / (precision + recall)


def confusion_rates(is_anomaly, predicted):
    """TPR, TNR, F1 и MCC. Положительный класс — аномалия."""
    is_anomaly = np.asarray(is_anomaly, dtype=bool)
    predicted = np.asarray(predicted, dtype=bool)
    true_positive = int(np.count_nonzero(is_anomaly & predicted))
    false_positive = int(np.count_nonzero(~is_anomaly & predicted))
    false_negative = int(np.count_nonzero(is_anomaly & ~predicted))
    true_negative = int(np.count_nonzero(~is_anomaly & ~predicted))
    tpr = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
    tnr = true_negative / (true_negative + false_positive) if true_negative + false_positive else 0.0
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
    f1 = 2.0 * precision * tpr / (precision + tpr) if precision + tpr else 0.0
    denominator = (
        (true_positive + false_positive)
        * (true_positive + false_negative)
        * (true_negative + false_positive)
        * (true_negative + false_negative)
    )
    if denominator == 0:
        mcc = 0.0
    else:
        mcc = (true_positive * true_negative - false_positive * false_negative) / np.sqrt(denominator)
    return {"tpr": float(tpr), "tnr": float(tnr), "f1": float(f1), "mcc": float(mcc)}


def remember_train_operating_point(distance_loss, f1, mcc):
    """Запоминает distance loss, F1 и MCC порога на всей обучающей выборке."""
    global LAST_TRAIN_DL, LAST_TRAIN_F1, LAST_TRAIN_MCC
    LAST_TRAIN_DL = float(distance_loss)
    LAST_TRAIN_F1 = float(f1)
    LAST_TRAIN_MCC = float(mcc)


def operating_record(pack, predicted, tau=0.0):
    """MCC, F1 и distance loss одной маски аномалий."""
    predicted = np.asarray(predicted, dtype=bool)
    loss, ratio = route_length_pair(pack, predicted)
    record = confusion_rates(pack["anom"], predicted)
    record.update(tau=float(tau), loss=float(loss), ratio=float(ratio))
    return record


def reference_route_length(pack):
    """Длина эталонного маршрута пакета, в метрах. Результат кэшируется."""
    cached = pack.get("route_ref_m")
    if cached is not None:
        return float(cached)
    truth = np.where(pack["anom"], 0.0, 1.0)
    length = CalculateMetrics.reference_route_length((pack["time"], pack["lon"], pack["lat"], truth))
    pack["route_ref_m"] = float(length)
    return pack["route_ref_m"]


def route_length_pair(pack, predicted_anomaly):
    """Distance loss и DLR маски аномалий. Та же длина, что в CalculateMetrics."""
    ref = reference_route_length(pack)
    predicted_anomaly = np.asarray(predicted_anomaly, dtype=bool)
    predicted_length = CalculateMetrics.path_length_m(
        pack["lat"][~predicted_anomaly], pack["lon"][~predicted_anomaly]
    )
    loss = abs(predicted_length - ref)
    ratio = loss / ref if ref > 0.0 else float("nan")
    return float(loss), float(ratio)


def pick_operating_point(records):
    """Максимум MCC. В полосе MCC_BAND — минимум distance loss."""
    usable = [
        record
        for record in records
        if np.isfinite(record["loss"]) and np.isfinite(record["mcc"])
    ]
    if not usable:
        raise RuntimeError("Нет конечного MCC и distance loss для выбора порога.")
    best_mcc = max(record["mcc"] for record in usable)
    band = [record for record in usable if record["mcc"] >= best_mcc - MCC_BAND - 1e-12]
    band.sort(key=lambda record: (record["loss"], -record["mcc"], -record["f1"], float(record.get("tau", 0.0))))
    return band[0], float(best_mcc), len(band)


def select_threshold(scores, pack, extra=()):
    """Ищет tau по квантилям score всей обучающей выборки.

    Большой score — более аномальная точка. Сначала максимизируется MCC.
    Среди порогов в полосе MCC_BAND берётся минимальный distance loss.
    """
    scores = np.nan_to_num(np.asarray(scores, dtype=float), nan=0.0, posinf=1e12, neginf=0.0)
    is_anomaly = np.asarray(pack["anom"], dtype=bool)
    if len(scores) != len(is_anomaly):
        raise ValueError("Длина score не совпадает с обучающей выборкой.")
    quantiles = np.quantile(scores, np.linspace(0.01, 0.995, 180))
    extras = np.atleast_1d(np.asarray(extra, dtype=float))
    candidates = np.unique(np.concatenate([quantiles, extras])) if extras.size else quantiles
    records = [operating_record(pack, scores > tau, tau) for tau in candidates]
    chosen, best_mcc, band_size = pick_operating_point(records)
    normal = scores[~is_anomaly]
    anomalous = scores[is_anomaly]
    if len(normal) and len(anomalous):
        print(f"  медиана score: move {np.median(normal):.4g}, аномалии {np.median(anomalous):.4g}")
    print(
        f"  выбран порог {chosen['tau']:.4g}: MCC {chosen['mcc']:.3f}, "
        f"лучший MCC {best_mcc:.3f}, в полосе {band_size}, "
        f"DL {chosen['loss']:.4g} м, F1 {chosen['f1']:.3f}, "
        f"доля move сохранена {chosen['tnr']:.1%}"
    )
    remember_train_operating_point(chosen["loss"], chosen["f1"], chosen["mcc"])
    return chosen["tau"], chosen["f1"]


def series_pair(predicted_anomaly):
    """Собирает prediction и ground truth для CalculateMetrics.

    Порядок кортежа — (время, долгота, широта, метка). Метка 0 — аномалия.
    """
    predicted_anomaly = np.asarray(predicted_anomaly, dtype=bool)
    truth = np.where(test["anom"], 0.0, 1.0)
    predicted = np.where(predicted_anomaly, 0.0, 1.0)
    ground_truth = (test["time"], test["lon"], test["lat"], truth)
    prediction = (test["time"], test["lon"].copy(), test["lat"].copy(), predicted)
    return prediction, ground_truth


def official_metrics(predicted_anomaly, anomaly_scores=None):
    """Считает метрики CalculateMetrics на тестовой выборке."""
    prediction, ground_truth = series_pair(predicted_anomaly)
    scores = None if anomaly_scores is None else np.asarray(anomaly_scores, dtype=float)
    return CalculateMetrics.calculate_all_metrics(prediction, ground_truth, scores)


RESULTS = []
LAST_TRAIN_DL = float("nan")
LAST_TRAIN_F1 = float("nan")
LAST_TRAIN_MCC = float("nan")
RATE_FIELDS = [
    ("point_precision", "Point precision"),
    ("point_recall", "Point recall"),
    ("point_f1", "Point F1"),
    ("pr_auc", "PR-AUC"),
    ("affiliation_precision", "Affiliation precision"),
    ("affiliation_recall", "Affiliation recall"),
    ("affiliation_f1", "Affiliation F1"),
]
LIST_FIELDS = [
    ("distance_loss", "Distance loss, м"),
    ("distance_loss_ratio", "Distance loss ratio"),
] + RATE_FIELDS + [
    ("detection_delay_median", "Delay median, с"),
    ("detection_delay_mean", "Delay mean, с"),
    ("detection_delay_p95", "Delay p95, с"),
    ("detection_delay_n_events", "Events"),
    ("detection_delay_n_detected", "Detected"),
    ("detection_delay_n_undetected", "Undetected"),
    ("geodesic_rmse", "Geodesic RMSE, м"),
    ("hausdorff_distance", "Hausdorff, м"),
]
COUNT_KEYS = {
    "detection_delay_n_events",
    "detection_delay_n_detected",
    "detection_delay_n_undetected",
}


def format_metric(key, value):
    if value is None or not np.isfinite(value):
        return f"{'—':>14}"
    return f"{format_metric_value(key, value):>14}"


def plot_metric_strip(name, metrics):
    labels = [label for _, label in RATE_FIELDS]
    values = []
    for key, _ in RATE_FIELDS:
        value = metrics[key]
        values.append(np.nan if not np.isfinite(value) else float(value))
    fig, ax = plt.subplots(figsize=(11.2, 1.45))
    image = np.ma.masked_invalid(np.asarray(values, dtype=float)[None, :])
    heat = ax.imshow(image, cmap="YlOrRd", vmin=0.0, vmax=1.0, aspect="auto")
    ax.set_yticks([0])
    ax.set_yticklabels([name])
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=25, ha="right")
    for index, value in enumerate(values):
        text = "—" if not np.isfinite(value) else f"{value:.3f}"
        ax.text(index, 0, text, ha="center", va="center", color="black", fontsize=9)
    fig.colorbar(heat, ax=ax, fraction=0.02, pad=0.02)
    fig.tight_layout()
    try:
        from IPython import get_ipython

        interactive = get_ipython() is not None
    except Exception:
        interactive = False
    if interactive:
        plt.show()
    else:
        plt.close(fig)


def slice_series(series, tail):
    """Берёт хвост ряда (время, долгота, широта, метка) для карты."""
    time, lon, lat, label = series
    return time[-tail:], lon[-tail:], lat[-tail:], label[-tail:]


def plot_classification_map(name, predicted_anomaly):
    """Карта ground truth и разметки модели по последним PLOT_TAIL точкам теста."""
    tail = min(PLOT_TAIL, len(predicted_anomaly))
    prediction, ground_truth = series_pair(predicted_anomaly)
    prediction = slice_series(prediction, tail)
    ground_truth = slice_series(ground_truth, tail)
    print(f"  Карта классификации: последние {nfmt(tail)} точек теста")
    map_kwargs = {"use_basemap": MAP_USE_BASEMAP}
    if MAP_BBOX is not None:
        map_kwargs["bbox"] = MAP_BBOX
    plot_sample_track(
        prediction,
        ground_truth,
        title=f"{name}: последние {nfmt(tail)} точек теста",
        figsize=(22.0, 11.0),
        annotate=False,
        **map_kwargs,
    )


def show_result(name, predicted_anomaly, elapsed, note, anomaly_scores=None):
    """Печатает список метрик CalculateMetrics, рисует полоску, карту и запоминает строку сводки."""
    global RESULTS, LAST_TRAIN_DL, LAST_TRAIN_F1, LAST_TRAIN_MCC
    predicted_anomaly = np.asarray(predicted_anomaly, dtype=bool)
    if predicted_anomaly.shape != test["anom"].shape:
        raise ValueError(f"{name}: длина маски {predicted_anomaly.shape[0]} вместо {test['anom'].shape[0]}")
    if anomaly_scores is not None and len(anomaly_scores) != len(predicted_anomaly):
        raise ValueError(f"{name}: длина score {len(anomaly_scores)} вместо {len(predicted_anomaly)}")
    metrics = official_metrics(predicted_anomaly, anomaly_scores)
    print("=" * 72)
    print(name)
    if note:
        print(note)
    print(f"  Помечено аномалиями на тесте: {predicted_anomaly.mean():.1%}   время {elapsed:.1f} с")
    print("  CalculateMetrics, положительный класс — аномалия (метка 0)")
    for key, label in LIST_FIELDS:
        print(f"    {label:<28} {format_metric(key, metrics[key]).strip():>14}")
    test_rates = confusion_rates(test["anom"], predicted_anomaly)
    print(f"    {'MCC':<28} {test_rates['mcc']:14.4f}")
    print(f"    {'Доля move сохранена':<28} {test_rates['tnr']:14.1%}")
    plot_metric_strip(name, metrics)
    plot_classification_map(name, predicted_anomaly)
    row = {
        "Метод": name,
        "Время, с": elapsed,
        "DL обучения, м": LAST_TRAIN_DL,
        "F1 обучения": LAST_TRAIN_F1,
        "MCC обучения": LAST_TRAIN_MCC,
        "mcc": test_rates["mcc"],
    }
    for key, _ in LIST_FIELDS:
        row[key] = metrics[key]
    RESULTS = [item for item in RESULTS if item["Метод"] != name]
    RESULTS.append(row)
    LAST_TRAIN_DL = float("nan")
    LAST_TRAIN_F1 = float("nan")
    LAST_TRAIN_MCC = float("nan")


def window_starts(pack, length, max_dt, max_dist, require_normal):
    """Индексы окон, которые не пересекают большой разрыв по времени или дистанции."""
    count = len(pack["lat"])
    if count < length:
        return np.empty(0, dtype=int)
    good = np.ones(count, dtype=bool)
    if require_normal:
        good &= pack["valid"]
    step_ok = np.ones(count, dtype=bool)
    step_ok[1:] = (pack["dt"][1:] <= max_dt) & (pack["dist"][1:] <= max_dist)
    good_cum = np.cumsum(good.astype(np.int32))
    step_cum = np.cumsum(step_ok.astype(np.int32))
    starts = np.arange(count - length + 1)
    good_sum = good_cum[starts + length - 1] - np.where(starts > 0, good_cum[starts - 1], 0)
    step_sum = step_cum[starts + length - 1] - step_cum[starts]
    return starts[(good_sum == length) & (step_sum == length - 1)]


def take_starts(starts, limit):
    if len(starts) <= limit:
        return starts
    picked = rng.choice(len(starts), size=limit, replace=False)
    return np.sort(starts[picked])


def scatter_max(count, starts, values, length):
    """Раскладывает оконные score по точкам, в пересечениях оставляет максимум."""
    scores = np.full(count, -np.inf)
    index = starts[:, None] + np.arange(length)[None, :]
    if np.ndim(values) == 1:
        payload = np.repeat(values, length)
    else:
        payload = np.asarray(values, dtype=float).ravel()
    np.maximum.at(scores, index.ravel(), payload)
    scores[~np.isfinite(scores)] = 0.0
    return scores


def train_torch(model, array, loss_fn, epochs, lr=1e-3, batch_size=256):
    model.to(DEVICE)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(np.ascontiguousarray(array))), batch_size=batch_size, shuffle=True
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        total, seen = 0.0, 0
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model, batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total += float(loss.detach()) * len(batch)
            seen += len(batch)
        print(f"    эпоха {epoch + 1}/{epochs}, loss {total / max(seen, 1):.5f}")
    model.eval()
    return model


def batched_numpy(model, array, forward, batch_size=512):
    model.eval()
    pieces = []
    with torch.no_grad():
        for start in range(0, len(array), batch_size):
            batch = torch.from_numpy(np.ascontiguousarray(array[start : start + batch_size])).to(DEVICE)
            pieces.append(forward(model, batch).detach().cpu().numpy())
    return np.concatenate(pieces, axis=0)


print(f"Начало локальной проекции: lat {origin_lat:.4f}, lon {origin_lon:.4f}")
print(
    f"Эталон маршрута: обучение {reference_route_length(fit):,.2f} м, "
    f"тест {reference_route_length(test):,.2f} м. "
    f"Полоса MCC — {MCC_BAND:.2f}: внутри неё порог с меньшим distance loss."
)

# Опорные разметки

Две постоянные маски задают края шкалы accuracy на этом тесте: ноль, если все точки объявлены штатными, и единица, если все объявлены аномальными.


In [ ]:
started = time.perf_counter()
show_result(
    "Всегда штатная",
    np.zeros(len(test["anom"]), dtype=bool),
    time.perf_counter() - started,
    "Каждая тестовая точка помечена штатной.",
)
started = time.perf_counter()
show_result(
    "Всегда аномалия",
    np.ones(len(test["anom"]), dtype=bool),
    time.perf_counter() - started,
    "Каждая тестовая точка помечена аномальной.",
)


# ARIMA(2, 1, 1)

Две независимые модели, по широте и по долготе. Коэффициенты оцениваются методом Хеннана–Риссанена на самом длинном гладком штатном участке обучения. На тесте — одношаговый прогноз; score — геодезическая ошибка, в метрах.


In [ ]:
def longest_smooth_slice(pack, max_dt=40.0, max_dist=2_000.0, min_len=80):
    good = pack["valid"].copy()
    good[1:] &= (pack["dt"][1:] <= max_dt) & (pack["dist"][1:] <= max_dist)
    padded = np.r_[False, good, False]
    change = np.diff(padded.astype(np.int8))
    starts = np.flatnonzero(change == 1)
    ends = np.flatnonzero(change == -1)
    if len(starts) == 0:
        raise RuntimeError("Нет гладкого штатного участка для ARIMA.")
    lengths = ends - starts
    best = int(np.argmax(lengths))
    if lengths[best] < min_len:
        raise RuntimeError("Самый длинный штатный участок короче 80 точек.")
    return int(starts[best]), int(ends[best])


def fit_arima_211(series):
    """Оценивает ARIMA(2, 1, 1): константу, два AR-коэффициента разностей и один MA."""
    series = np.asarray(series, dtype=float)
    delta = np.diff(series)
    constant = float(np.mean(delta))
    centered = delta - constant
    long_order = 12
    rows = len(centered) - long_order
    design = np.column_stack([centered[long_order - lag : len(centered) - lag] for lag in range(1, long_order + 1)])
    target = centered[long_order:]
    coef, *_ = np.linalg.lstsq(design, target, rcond=None)
    residual = np.zeros(len(centered))
    residual[long_order:] = target - design @ coef
    start = long_order + 1
    target2 = centered[start:]
    design2 = np.column_stack(
        [
            centered[start - 1 : len(centered) - 1],
            centered[start - 2 : len(centered) - 2],
            residual[start - 1 : len(centered) - 1],
        ]
    )
    beta, *_ = np.linalg.lstsq(design2, target2, rcond=None)
    phi1, phi2, theta = (float(item) for item in beta)
    while abs(phi1) > 1.8 or abs(phi2) > 0.95:
        phi1 *= 0.8
        phi2 *= 0.8
    return {"c": constant, "phi1": phi1, "phi2": phi2, "theta": theta}


def arima_forecast(series, params):
    """Одношаговый прогноз ARIMA(2, 1, 1) с обновлением по фактическому приращению."""
    predicted = np.empty(len(series))
    predicted[0] = series[0]
    prev_delta_1 = 0.0
    prev_delta_2 = 0.0
    prev_eps = 0.0
    for index in range(1, len(series)):
        delta_hat = (
            params["c"] + params["phi1"] * prev_delta_1 + params["phi2"] * prev_delta_2 + params["theta"] * prev_eps
        )
        predicted[index] = series[index - 1] + delta_hat
        delta = series[index] - series[index - 1]
        prev_eps = delta - delta_hat
        prev_delta_2 = prev_delta_1
        prev_delta_1 = delta
    return predicted


def arima_scores(pack, lat_params, lon_params):
    lat_hat = arima_forecast(pack["lat"], lat_params)
    lon_hat = arima_forecast(pack["lon"], lon_params)
    return haversine_m(pack["lat"], pack["lon"], lat_hat, lon_hat)


print("ARIMA(2, 1, 1)")
started = time.perf_counter()
span_start, span_end = longest_smooth_slice(fit)
print(f"  участок обучения: точки {span_start}–{span_end}, длина {span_end - span_start}")
lat_params = fit_arima_211(fit["lat"][span_start:span_end])
lon_params = fit_arima_211(fit["lon"][span_start:span_end])
print(
    f"  широта  c={lat_params['c']:.3g} φ1={lat_params['phi1']:.3f} φ2={lat_params['phi2']:.3f} θ={lat_params['theta']:.3f}"
)
print(
    f"  долгота c={lon_params['c']:.3g} φ1={lon_params['phi1']:.3f} φ2={lon_params['phi2']:.3f} θ={lon_params['theta']:.3f}"
)
fit_scores = arima_scores(fit, lat_params, lon_params)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = arima_scores(test, lat_params, lon_params)
print(f"  порог {tau:.4g} м, DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}")
show_result(
    "ARIMA(2, 1, 1)",
    test_scores > tau,
    time.perf_counter() - started,
    f"Порог геодезической ошибки {tau:.1f} м. DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}.",
    test_scores,
)


# Фильтр Калмана, постоянная скорость

Состояние — положение и скорость в локальной проекции. Score — нормированный квадрат инновации, при согласованной модели он имеет распределение хи-квадрат с двумя степенями свободы. После разрыва дольше 180 секунд фильтр инициализируется заново, но сама инновация этого шага сохраняется.


In [ ]:
def process_noise(dt, sigma_acc):
    dt2 = dt * dt
    dt3 = dt2 * dt
    dt4 = dt2 * dt2
    noise = np.zeros((4, 4))
    noise[0, 0] = noise[1, 1] = dt4 / 4.0
    noise[2, 2] = noise[3, 3] = dt2
    noise[0, 2] = noise[2, 0] = noise[1, 3] = noise[3, 1] = dt3 / 2.0
    return noise * (sigma_acc**2)


def kalman_scores(pack, sigma_acc=0.35, sigma_meas=40.0):
    xy = pack["xy"]
    t_sec = pack["t"]
    count = len(xy)
    state = np.zeros(4)
    state[:2] = xy[0]
    cov = np.diag([80.0**2, 80.0**2, 5.0**2, 5.0**2])
    observation = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]])
    meas_cov = np.eye(2) * (sigma_meas**2)
    eye = np.eye(4)
    scores = np.zeros(count)
    for index in range(1, count):
        dt = float(t_sec[index] - t_sec[index - 1])
        dt = 1.0 if dt <= 0.0 else dt
        gap = dt > 180.0
        dt_model = min(dt, 45.0)
        transition = np.eye(4)
        transition[0, 2] = dt_model
        transition[1, 3] = dt_model
        pred = transition @ state
        pred_cov = transition @ cov @ transition.T + process_noise(dt_model, sigma_acc)
        innov = xy[index] - observation @ pred
        innov_cov = observation @ pred_cov @ observation.T + meas_cov
        solved = np.linalg.solve(innov_cov, innov)
        scores[index] = float(innov @ solved)
        gain = pred_cov @ observation.T @ np.linalg.inv(innov_cov)
        state = pred + gain @ innov
        cov = (eye - gain @ observation) @ pred_cov
        cov = 0.5 * (cov + cov.T)
        if gap:
            state = np.array([xy[index, 0], xy[index, 1], 0.0, 0.0])
            cov = np.diag([80.0**2, 80.0**2, 5.0**2, 5.0**2])
    return scores


print("Фильтр Калмана")
started = time.perf_counter()
chi_critical = float(chi2.ppf(0.999, 2))
fit_scores = kalman_scores(fit)
tau, train_f1 = select_threshold(fit_scores, fit, extra=(chi_critical,))
test_scores = kalman_scores(test)
print(f"  χ² 0.999 (df=2) = {chi_critical:.2f}; выбранный порог {tau:.3g}, DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}")
show_result(
    "Фильтр Калмана",
    test_scores > tau,
    time.perf_counter() - started,
    f"Порог NIS {tau:.2f}. Теоретический χ²₀.₉₉₉ = {chi_critical:.2f}. DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}.",
    test_scores,
)


# Bootstrap particle filter (SIR)

Те же положение и скорость, но вместо одного гауссовского состояния держится набор гипотез. После ресэмплинга веса равны, и score — минус логарифм средней правдоподобности измерения.


In [ ]:
def systematic_resample(log_weights, generator):
    shifted = np.exp(log_weights - np.max(log_weights))
    weights = shifted / shifted.sum()
    positions = (np.arange(len(weights)) + generator.random()) / len(weights)
    cumulative = np.cumsum(weights)
    cumulative[-1] = 1.0
    return np.searchsorted(cumulative, positions, side="left").clip(0, len(weights) - 1)


def particle_scores(pack, n_particles=N_PARTICLES, sigma_acc=0.45, sigma_meas=40.0, seed=SEED):
    generator = np.random.default_rng(seed)
    xy = pack["xy"]
    t_sec = pack["t"]
    count = len(xy)
    particles = np.zeros((n_particles, 4))
    particles[:, :2] = xy[0]
    particles[:, 2:] = generator.normal(0.0, 1.0, size=(n_particles, 2))
    scores = np.zeros(count)
    r_var = sigma_meas**2
    log_const = np.log(2.0 * np.pi * r_var)
    for index in range(1, count):
        dt = float(t_sec[index] - t_sec[index - 1])
        dt = 1.0 if dt <= 0.0 else dt
        gap = dt > 180.0
        dt_model = min(dt, 45.0)
        particles[:, 0] += particles[:, 2] * dt_model
        particles[:, 1] += particles[:, 3] * dt_model
        particles[:, 0] += generator.normal(0.0, sigma_acc * dt_model**2 / 2.0, n_particles)
        particles[:, 1] += generator.normal(0.0, sigma_acc * dt_model**2 / 2.0, n_particles)
        particles[:, 2] += generator.normal(0.0, sigma_acc * dt_model, n_particles)
        particles[:, 3] += generator.normal(0.0, sigma_acc * dt_model, n_particles)
        diff = xy[index] - particles[:, :2]
        loglik = -0.5 * (np.sum(diff * diff, axis=1) / r_var + 2.0 * log_const)
        scores[index] = float(-logsumexp(loglik) + np.log(n_particles))
        draw = systematic_resample(loglik, generator)
        particles = particles[draw]
        if gap or scores[index] > 40.0:
            particles[:, :2] = xy[index]
            particles[:, 2:] = generator.normal(0.0, 1.0, size=(n_particles, 2))
    return scores


print("Particle filter")
started = time.perf_counter()
fit_scores = particle_scores(fit)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = particle_scores(test)
print(f"  частиц {N_PARTICLES}, порог {tau:.3g}, DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}")
show_result(
    "Particle filter",
    test_scores > tau,
    time.perf_counter() - started,
    f"SIR, частиц {N_PARTICLES}. Порог {tau:.2f}. DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}.",
    test_scores,
)


# Gaussian process

Два независимых GP с RBF-ядром описывают восточный и северный шаг в метрах, а не положение на карте. Опорные 250 точек — `move` обучения. На участке, далёком по времени от обучения, прогноз сходится к среднему штатному шагу, поэтому другой район сам по себе score не поднимает. Score — сумма квадратов нормированных остатков шага.



In [ ]:
def rbf(left, right, lengthscale, signal):
    gap = left.reshape(-1, 1) - right.reshape(1, -1)
    return (signal**2) * np.exp(-0.5 * gap * gap / lengthscale**2)


def gp_predict(t_train, y_train, t_test, lengthscale, noise, signal):
    center = float(np.mean(y_train))
    target = y_train - center
    chol = None
    jitter = 1e-4 * signal**2
    for _ in range(6):
        gram = rbf(t_train, t_train, lengthscale, signal)
        gram.flat[:: len(t_train) + 1] += noise**2 + jitter
        try:
            chol = np.linalg.cholesky(gram)
            break
        except np.linalg.LinAlgError:
            jitter *= 10.0
    if chol is None:
        raise np.linalg.LinAlgError("Ковариация GP не разложилась.")
    alpha = np.linalg.solve(chol.T, np.linalg.solve(chol, target))
    mean = np.empty(len(t_test))
    variance = np.empty(len(t_test))
    batch = 2_500
    for start in range(0, len(t_test), batch):
        chunk = t_test[start : start + batch]
        cross = rbf(t_train, chunk, lengthscale, signal)
        mean[start : start + len(chunk)] = cross.T @ alpha + center
        solved = np.linalg.solve(chol, cross)
        variance[start : start + len(chunk)] = signal**2 + noise**2 - np.sum(solved * solved, axis=0)
    return mean, np.clip(variance, 1e-6, None)


def gp_scores(pack, t_train, x_train, y_train, lengthscale, noise):
    signal_x = float(np.std(x_train) + 1.0)
    signal_y = float(np.std(y_train) + 1.0)
    mean_x, var_x = gp_predict(t_train, x_train, pack["t"], lengthscale, noise, signal_x)
    mean_y, var_y = gp_predict(t_train, y_train, pack["t"], lengthscale, noise, signal_y)
    return (pack["dx"] - mean_x) ** 2 / var_x + (pack["dy"] - mean_y) ** 2 / var_y


print("Gaussian process")
started = time.perf_counter()
normal_index = np.flatnonzero(fit["valid"])
gp_count = 250
grid = np.linspace(0, len(normal_index) - 1, gp_count).astype(int)
chosen = normal_index[grid]
t_gp = fit["t"][chosen]
x_gp = fit["dx"][chosen]
y_gp = fit["dy"][chosen]
lengthscale = 6.0 * 3600.0
step_scale = float(np.median(np.hypot(x_gp, y_gp)))
noise = max(5.0, 0.25 * step_scale)
fit_scores = gp_scores(fit, t_gp, x_gp, y_gp, lengthscale, noise)
tau, train_f1 = select_threshold(fit_scores, fit, extra=(chi2.ppf(0.999, 2),))
test_scores = gp_scores(test, t_gp, x_gp, y_gp, lengthscale, noise)
print(
    f"  опорных точек {gp_count}, длина корреляции {lengthscale / 3600:.1f} ч, шум шага {noise:.1f} м, "
    f"порог {tau:.3g}, DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}"
)
show_result(
    "Gaussian process",
    test_scores > tau,
    time.perf_counter() - started,
    f"RBF по шагу dx, dy, опорных точек move {gp_count}, ℓ = 6 ч. Порог {tau:.2f}. "
    f"DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}.",
    test_scores,
)






# GP-EKF, GP-BayesFilters

Ko и Fox подставляют гауссовский процесс в расширенный фильтр Калмана: процесс учит переход состояния, якобиан среднего заменяет матрицу линеаризации. У трека нет управления. Параметрическая часть — постоянная скорость в локальной проекции, GP учит остаток этого перехода по скорости и шагу времени на штатных парах обучения. Наблюдение — линейная модель ГНСС, не второй GP: измерение и так является положением. Score — нормированный квадрат инновации. После разрыва дольше 180 секунд фильтр инициализируется заново, инновация этого шага сохраняется.

In [ ]:
from app.metrics.gp_bayes_filter import GpEkfModel

print("GP-EKF")
started = time.perf_counter()
gp_ekf = GpEkfModel(support=160, seed=SEED, sigma_meas=40.0).fit(fit["xy"], fit["t"], fit["valid"])
fit_scores = gp_ekf.scores(fit["xy"], fit["t"])
tau, train_f1 = select_threshold(fit_scores, fit, extra=(chi2.ppf(0.999, 2),))
test_scores = gp_ekf.scores(test["xy"], test["t"])
print(
    f"  опорных переходов {gp_ekf.n_support}, длина корреляции {gp_ekf.lengthscale:.2g}, шум {gp_ekf.noise:.2g}, "
    f"порог {tau:.3g}, DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}"
)
show_result(
    "GP-EKF",
    test_scores > tau,
    time.perf_counter() - started,
    (
        f"GP-BayesFilters, остаток постоянной скорости. Опорных переходов {gp_ekf.n_support}, "
        f"длина корреляции {gp_ekf.lengthscale:.2g}, шум {gp_ekf.noise:.2g}. Порог NIS {tau:.2f}. "
        f"DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}."
    ),
    test_scores,
)

# Скрытая марковская модель

Наблюдение точки — скорость, ускорение и поворот курса. Одна гауссовская HMM учится на штатных отрезках, две — на аномальных: общий режим и быстрые участки. Point-wise score — максимум разности предсказательных логарифмов правдоподобия аномальной и штатной модели.


In [ ]:
class DiagHMM:
    """HMM с диагональными гауссовскими эмиссиями и несколькими последовательностями."""

    def __init__(self, n_states=3, n_iter=8, seed=SEED):
        self.n_states = n_states
        self.n_iter = n_iter
        self.rng = np.random.default_rng(seed)

    def _log_emit(self, values):
        diff = values[:, None, :] - self.mean[None, :, :]
        quad = np.sum(diff * diff / self.var[None, :, :], axis=-1)
        logdet = np.sum(np.log(2.0 * np.pi * self.var), axis=-1)
        return -0.5 * (quad + logdet)

    def _forward(self, log_emit):
        length, _ = log_emit.shape
        log_alpha = np.empty_like(log_emit)
        log_trans = np.log(self.trans + 1e-16)
        log_alpha[0] = np.log(self.start + 1e-16) + log_emit[0]
        for index in range(1, length):
            log_alpha[index] = log_emit[index] + logsumexp(log_alpha[index - 1][:, None] + log_trans, axis=0)
        return log_alpha

    def _backward(self, log_emit):
        length, _ = log_emit.shape
        log_beta = np.zeros_like(log_emit)
        log_trans = np.log(self.trans + 1e-16)
        for index in range(length - 2, -1, -1):
            log_beta[index] = logsumexp(
                log_trans + log_emit[index + 1][None, :] + log_beta[index + 1][None, :],
                axis=1,
            )
        return log_beta

    def fit(self, sequences):
        stacked = np.concatenate(sequences, axis=0)
        width = stacked.shape[1]
        levels = np.linspace(0.2, 0.8, self.n_states)
        self.mean = np.quantile(stacked, levels, axis=0)
        self.mean = self.mean + self.rng.normal(0.0, 0.01, size=self.mean.shape)
        base_var = np.var(stacked, axis=0) + 1e-2
        self.var = np.repeat(base_var[None, :], self.n_states, axis=0)
        self.trans = np.full((self.n_states, self.n_states), 1.0 / self.n_states)
        self.start = np.full(self.n_states, 1.0 / self.n_states)
        for _ in range(self.n_iter):
            start_acc = np.zeros(self.n_states)
            trans_acc = np.zeros_like(self.trans)
            mean_acc = np.zeros((self.n_states, width))
            var_acc = np.zeros((self.n_states, width))
            weight = np.zeros(self.n_states)
            for seq in sequences:
                log_emit = self._log_emit(seq)
                log_alpha = self._forward(log_emit)
                log_beta = self._backward(log_emit)
                log_gamma = log_alpha + log_beta
                log_gamma -= logsumexp(log_gamma, axis=1, keepdims=True)
                gamma = np.exp(log_gamma)
                log_xi = (
                    log_alpha[:-1, :, None]
                    + np.log(self.trans + 1e-16)[None, :, :]
                    + log_emit[1:, None, :]
                    + log_beta[1:, None, :]
                )
                log_xi -= logsumexp(log_xi.reshape(len(seq) - 1, -1), axis=1)[:, None, None]
                start_acc += gamma[0]
                trans_acc += np.exp(log_xi).sum(axis=0)
                weight += gamma.sum(axis=0)
                mean_acc += gamma.T @ seq
                for state in range(self.n_states):
                    diff = seq - self.mean[state]
                    var_acc[state] += (gamma[:, state : state + 1] * (diff * diff)).sum(axis=0)
            self.start = start_acc / start_acc.sum()
            self.trans = trans_acc / np.maximum(trans_acc.sum(axis=1, keepdims=True), 1e-12)
            self.mean = mean_acc / np.maximum(weight[:, None], 1e-8)
            self.var = var_acc / np.maximum(weight[:, None], 1e-8) + 1e-2
        return self

    def predictive_loglik(self, values):
        log_emit = self._log_emit(values)
        log_trans = np.log(self.trans + 1e-16)
        log_filt = np.log(self.start + 1e-16)
        output = np.empty(len(values))
        for index in range(len(values)):
            log_pred = log_filt if index == 0 else logsumexp(log_filt[:, None] + log_trans, axis=0)
            joint = log_pred + log_emit[index]
            output[index] = logsumexp(joint)
            log_filt = joint - output[index]
        return output


def class_sequences(pack, mask, max_sequences=6, max_len=1_200, min_len=30, max_dt=90.0):
    flag = mask.copy()
    flag[1:] &= pack["dt"][1:] <= max_dt
    padded = np.r_[False, flag, False]
    change = np.diff(padded.astype(np.int8))
    starts = np.flatnonzero(change == 1)
    ends = np.flatnonzero(change == -1)
    order = np.argsort(-(ends - starts))
    sequences = []
    for index in order:
        if ends[index] - starts[index] < min_len:
            break
        stop = min(ends[index], starts[index] + max_len)
        sequences.append(kinematic_matrix(pack, slice(int(starts[index]), int(stop))))
        if len(sequences) >= max_sequences:
            break
    return sequences


def hmm_scores(pack, normal_model, anomaly_models):
    values = kinematic_matrix(pack)
    normal_log = normal_model.predictive_loglik(values)
    anomaly_log = np.vstack([model.predictive_loglik(values) for model in anomaly_models])
    return anomaly_log.max(axis=0) - normal_log


print("HMM")
started = time.perf_counter()
normal_seqs = class_sequences(fit, fit["valid"])
anomaly_seqs = class_sequences(fit, fit["anom"])
fast_seqs = class_sequences(fit, fit["anom"] & (fit["speed"] >= 8.0), min_len=20)
print(f"  штатных отрезков {len(normal_seqs)}, аномальных {len(anomaly_seqs)}, быстрых {len(fast_seqs)}")
if not normal_seqs or not anomaly_seqs:
    raise RuntimeError("Для HMM не нашлось отрезков обоих классов.")
iterations = 8
normal_model = DiagHMM(n_states=3, n_iter=iterations, seed=SEED).fit(normal_seqs)
anomaly_models = [DiagHMM(n_states=3, n_iter=iterations, seed=SEED + 1).fit(anomaly_seqs)]
if fast_seqs:
    anomaly_models.append(DiagHMM(n_states=2, n_iter=iterations, seed=SEED + 2).fit(fast_seqs))
fit_scores = hmm_scores(fit, normal_model, anomaly_models)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = hmm_scores(test, normal_model, anomaly_models)
print(f"  порог {tau:.3g}, DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}")
show_result(
    "HMM",
    test_scores > tau,
    time.perf_counter() - started,
    f"Гауссовские HMM, аномальных моделей {len(anomaly_models)}. Порог {tau:.2f}. DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}.",
    test_scores,
)


# Фильтр Хэмпела

В центрированном окне скорость сравнивается с медианой соседей. Score — отклонение в единицах MAD. Окно видит будущие точки: это пакетный вариант фильтра, не онлайн-поток.


In [ ]:
def hampel_scores(speed, half_window=10):
    padded = np.pad(speed, half_window, mode="edge")
    windows = np.lib.stride_tricks.sliding_window_view(padded, 2 * half_window + 1)
    center = np.median(windows, axis=1)
    mad = np.median(np.abs(windows - center[:, None]), axis=1)
    return np.abs(speed - center) / (1.4826 * mad + 1e-9)


print("Hampel")
started = time.perf_counter()
fit_scores = hampel_scores(fit["speed"])
tau, train_f1 = select_threshold(fit_scores, fit, extra=(3.0, 3.5, 5.0))
test_scores = hampel_scores(test["speed"])
print(f"  полуокно 10, порог {tau:.3g}, DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}")
show_result(
    "Hampel",
    test_scores > tau,
    time.perf_counter() - started,
    f"Окно 21 точка по скорости. Порог {tau:.2f}. DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}.",
    test_scores,
)


# DBSCAN

Признаки точки — восточный и северный шаг, скорость, ускорение, поворот и логарифм формы окна. Абсолютные координаты не используются, поэтому кластер штатного хода не привязан к району обучения. Кластеры строятся по всему тестовому набору: шумовая метка `-1` и есть аномалия. Радиус `eps` подбирается по всей обучающей выборке: максимальный MCC, а в полосе `MCC_BAND` — минимальный distance loss. Шкала признаков — робастная, по точкам `move` обучения, чтобы `eps` на обучении и на тесте был в одних единицах.

Непрерывный score для PR-AUC — адаптация поверх меток DBSCAN: максимум из k-distance до `MinPts`-го соседа и расстояния до ближайшей core-точки. Большее значение — точка сильнее изолирована от плотного режима движения.



In [ ]:
from sklearn.cluster import DBSCAN


def density_matrix(pack):
    return local_motion_matrix(pack)


def dbscan_anomaly_scores(features, min_samples, labels=None, core_indices=None):
    """k-distance и расстояние до ближайшей core-точки. Больше — более аномально."""
    neighbors = NearestNeighbors(n_neighbors=min_samples, algorithm="kd_tree").fit(features)
    k_distance = neighbors.kneighbors(features)[0][:, -1]
    if core_indices is None or len(core_indices) == 0:
        return k_distance
    core_nn = NearestNeighbors(n_neighbors=1, algorithm="kd_tree").fit(features[core_indices])
    core_distance = core_nn.kneighbors(features)[0][:, 0]
    return np.maximum(k_distance, core_distance)


print("DBSCAN")
started = time.perf_counter()
center, scale = robust_params(density_matrix(fit)[fit["valid"]])
fit_x = apply_robust(density_matrix(fit), center, scale)
test_x = apply_robust(density_matrix(test), center, scale)
min_samples = 8
neighbors = NearestNeighbors(n_neighbors=min_samples, algorithm="kd_tree").fit(fit_x)
distances, _ = neighbors.kneighbors(fit_x)
k_distance = np.sort(distances[:, -1])
eps_grid = np.unique(np.quantile(k_distance, [0.5, 0.7, 0.85, 0.93, 0.97]))
records = []
for eps in eps_grid:
    labels = DBSCAN(eps=float(eps), min_samples=min_samples, algorithm="kd_tree", n_jobs=-1).fit_predict(fit_x)
    predicted = labels == -1
    record = operating_record(fit, predicted, eps)
    record["eps"] = float(eps)
    print(
        f"  eps {eps:.3f}: доля шума {predicted.mean():.1%}, "
        f"MCC {record['mcc']:.3f}, DL {record['loss']:.4g} м, F1 {record['f1']:.3f}"
    )
    records.append(record)
chosen, best_mcc, band_size = pick_operating_point(records)
best_eps = chosen["eps"]
best_f1 = chosen["f1"]
print(
    f"  выбран eps {best_eps:.3f}: MCC {chosen['mcc']:.3f}, лучший MCC {best_mcc:.3f}, "
    f"в полосе {band_size}, DL {chosen['loss']:.4g} м, F1 {best_f1:.3f}"
)
model = DBSCAN(eps=best_eps, min_samples=min_samples, algorithm="kd_tree", n_jobs=-1)
test_labels = model.fit_predict(test_x)
test_scores = dbscan_anomaly_scores(
    test_x,
    min_samples,
    labels=test_labels,
    core_indices=model.core_sample_indices_,
)
print(
    f"  медиана score: шум {np.median(test_scores[test_labels == -1]) if np.any(test_labels == -1) else float('nan'):.4g}, "
    f"кластер {np.median(test_scores[test_labels != -1]) if np.any(test_labels != -1) else float('nan'):.4g}"
)
remember_train_operating_point(chosen["loss"], best_f1, chosen["mcc"])
show_result(
    "DBSCAN",
    test_labels == -1,
    time.perf_counter() - started,
    (
        f"eps {best_eps:.3f}, min_samples {min_samples}. "
        "Score = max(k-distance, расстояние до core). "
        f"DL обучения {LAST_TRAIN_DL:.4g} м, F1 {best_f1:.3f}."
    ),
    test_scores,
)





# Local Outlier Factor

Признаки — шаг по востоку и северу, скорость, ускорение и поворот. LOF считается по тестовому набору. Порог переносится со всей обучающей выборки, где LOF посчитан отдельно.


In [ ]:
def lof_matrix(pack):
    return np.column_stack([pack["dx"], pack["dy"], pack["speed"], pack["acc"], pack["dpsi"]])


def lof_scores(matrix, n_neighbors=35):
    model = LocalOutlierFactor(n_neighbors=n_neighbors, algorithm="kd_tree", n_jobs=-1)
    # Совпадающие координаты ломают оценку плотности, поэтому точки чуть разводятся.
    jittered = np.asarray(matrix, dtype=float) + np.random.default_rng(SEED).normal(0.0, 1e-8, size=np.shape(matrix))
    model.fit_predict(jittered)
    return -model.negative_outlier_factor_


print("LOF")
started = time.perf_counter()
lof_center, lof_scale = robust_params(lof_matrix(fit)[fit["valid"]])
fit_scores = lof_scores(apply_robust(lof_matrix(fit), lof_center, lof_scale))
tau, train_f1 = select_threshold(fit_scores, fit, extra=(1.5, 2.0))
test_scores = lof_scores(apply_robust(lof_matrix(test), lof_center, lof_scale))
print(f"  соседей 35, порог {tau:.3g}, DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}")
show_result(
    "LOF",
    test_scores > tau,
    time.perf_counter() - started,
    f"k = 35. Порог LOF {tau:.2f}. DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}.",
    test_scores,
)


# Isolation Forest

Один лес строится по обучающим признакам и тем же лесом оценивается тест. Признаки — шаг, скорость, ускорение, поворот и логарифм формы окна, без абсолютных координат. Score — противоположный `decision_function` sklearn, чтобы большее значение означало более аномальную точку.



In [ ]:
def forest_matrix(pack):
    return local_motion_matrix(pack)


print("Isolation Forest")
started = time.perf_counter()
forest_center, forest_scale = robust_params(forest_matrix(fit)[fit["valid"]])
fit_x = apply_robust(forest_matrix(fit), forest_center, forest_scale)
test_x = apply_robust(forest_matrix(test), forest_center, forest_scale)
forest = IsolationForest(
    n_estimators=200,
    max_samples=256,
    contamination="auto",
    random_state=SEED,
    n_jobs=-1,
)
forest.fit(fit_x)
fit_scores = -forest.decision_function(fit_x)
tau, train_f1 = select_threshold(fit_scores, fit, extra=(0.0,))
test_scores = -forest.decision_function(test_x)
print(f"  порог {tau:.3g}, DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}")
show_result(
    "Isolation Forest",
    test_scores > tau,
    time.perf_counter() - started,
    f"200 деревьев, max_samples 256. Порог {tau:.3f}. DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}.",
    test_scores,
)



# One-Class SVM

Область носителя оценивается только по точкам `move` обучения. Признаки — шаг, скорость, ускорение, поворот и логарифм формы окна, без абсолютных координат. Аномалия — точка с `f(x) < 0`, поэтому порог score `-f` фиксирован нулём. Параметр `nu` выбирается по MCC на всей обучающей выборке; в полосе `MCC_BAND` берётся меньший distance loss.



In [ ]:
def svm_matrix(pack):
    return local_motion_matrix(pack)


print("One-Class SVM")
started = time.perf_counter()
svm_raw = svm_matrix(fit)
svm_center, svm_scale = robust_params(svm_raw[fit["valid"]])
normal_x = apply_robust(svm_raw[fit["valid"]], svm_center, svm_scale)
sample_size = min(len(normal_x), 4_000)
sample_index = rng.choice(len(normal_x), size=sample_size, replace=False)
train_x = normal_x[sample_index]
fit_x = apply_robust(svm_matrix(fit), svm_center, svm_scale)
test_x = apply_robust(svm_matrix(test), svm_center, svm_scale)
records = []
for nu in (0.01, 0.05, 0.1):
    model = OneClassSVM(kernel="rbf", gamma="scale", nu=nu)
    model.fit(train_x)
    predicted = -model.decision_function(fit_x) > 0.0
    record = operating_record(fit, predicted, nu)
    record["model"] = model
    print(f"  nu {nu:.2f}: MCC {record['mcc']:.3f}, DL {record['loss']:.4g} м, F1 {record['f1']:.3f}")
    records.append(record)
chosen, best_mcc, band_size = pick_operating_point(records)
best_model = chosen["model"]
best_f1 = chosen["f1"]
print(
    f"  выбран nu {chosen['tau']:.2f}: MCC {chosen['mcc']:.3f}, лучший MCC {best_mcc:.3f}, "
    f"в полосе {band_size}, DL {chosen['loss']:.4g} м, F1 {best_f1:.3f}"
)
test_scores = -best_model.decision_function(test_x)
remember_train_operating_point(chosen["loss"], best_f1, chosen["mcc"])
show_result(
    "One-Class SVM",
    test_scores > 0.0,
    time.perf_counter() - started,
    (
        f"RBF, nu {chosen['tau']:.2f}, обучение на {nfmt(sample_size)} точках move. "
        f"Порог score фиксирован нулём. DL обучения {LAST_TRAIN_DL:.4g} м, F1 {best_f1:.3f}."
    ),
    test_scores,
)





# Robust PCA

Матрица собирается из окон кинематических признаков оцениваемой траектории и раскладывается inexact ALM на низкоранговую и разреженную части. Score точки — максимум нормы разреженного вектора этой точки по окнам, которые её содержат.


In [ ]:
def soft_threshold(matrix, amount):
    return np.sign(matrix) * np.maximum(np.abs(matrix) - amount, 0.0)


def inexact_alm_rpca(data, lam=None, max_iter=30, tol=1e-5):
    """Principal Component Pursuit: min ||L||_* + λ||S||_1 при X = L + S."""
    work = np.asarray(data, dtype=float)
    rows, cols = work.shape
    if lam is None:
        lam = 1.0 / np.sqrt(max(rows, cols))
    spectral = np.linalg.norm(work, 2)
    mu = 1.25 / max(spectral, 1e-8)
    mu_max = mu * 1e7
    factor = 1.6
    lagrange = work / max(spectral, np.max(np.abs(work)) / lam)
    sparse = np.zeros_like(work)
    low_rank = np.zeros_like(work)
    for _ in range(max_iter):
        basis, singular, right = np.linalg.svd(work - sparse + lagrange / mu, full_matrices=False)
        low_rank = (basis * np.maximum(singular - 1.0 / mu, 0.0)) @ right
        residual = work - low_rank + lagrange / mu
        sparse = soft_threshold(residual, lam / mu)
        error = work - low_rank - sparse
        lagrange = lagrange + mu * error
        if np.linalg.norm(error, "fro") / max(np.linalg.norm(work, "fro"), 1e-8) < tol:
            break
        mu = min(mu * factor, mu_max)
    return low_rank, sparse


def rpca_point_scores(pack, length=16, stride=8):
    features = np.column_stack([pack["dx"] / 1000.0, pack["dy"] / 1000.0, pack["speed"], pack["dpsi"]])
    starts = window_starts(pack, length, max_dt=180.0, max_dist=np.inf, require_normal=False)
    starts = starts[::stride]
    if len(starts) == 0:
        return np.zeros(len(pack["lat"]))
    index = starts[:, None] + np.arange(length)[None, :]
    matrix = features[index].reshape(len(starts), -1)
    center, scale = robust_params(matrix)
    _, sparse = inexact_alm_rpca(apply_robust(matrix, center, scale))
    row_score = np.linalg.norm(sparse.reshape(len(starts), length, -1), axis=2)
    return scatter_max(len(pack["lat"]), starts, row_score, length)


print("Robust PCA")
started = time.perf_counter()
fit_scores = rpca_point_scores(fit)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = rpca_point_scores(test)
print(f"  порог {tau:.3g}, DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}")
show_result(
    "Robust PCA",
    test_scores > tau,
    time.perf_counter() - started,
    f"Окна длины 16, λ = 1/√max(n, m). Порог {tau:.3f}. DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}.",
    test_scores,
)


# LSTM-автоэнкодер

Окна локальной траектории в километрах кодируются LSTM и восстанавливаются из одного скрытого состояния, без подачи входа в декодер. Score окна — диагональное расстояние Махаланобиса вектора ошибки. Параметры распределения ошибки оцениваются на штатных окнах, которые не входили в обучение. Точке присваивается максимум по покрывающим её окнам.


In [ ]:
class LSTMAutoencoder(nn.Module):
    """Encoder-decoder: вся последовательность восстанавливается из финального состояния."""

    def __init__(self, n_features=2, hidden=8):
        super().__init__()
        self.encoder = nn.LSTM(n_features, hidden, batch_first=True)
        self.decoder = nn.LSTM(n_features, hidden, batch_first=True)
        self.proj = nn.Linear(hidden, n_features)

    def forward(self, sequence):
        _, state = self.encoder(sequence)
        decoded, _ = self.decoder(torch.zeros_like(sequence), state)
        return self.proj(decoded)


def relative_xy_km(pack, starts, length):
    index = starts[:, None] + np.arange(length)[None, :]
    block = pack["xy"][index]
    return ((block - block[:, :1, :]) / 1000.0).astype(np.float32)


def split_starts(starts, train_fraction=0.85):
    order = rng.permutation(len(starts))
    cut = max(1, int(len(starts) * train_fraction))
    return starts[order[:cut]], starts[order[cut:]]


def diagonal_mahalanobis(error, mean, var):
    return np.sum((error - mean) ** 2 / var, axis=1)


print("LSTM-автоэнкодер")
started = time.perf_counter()
lstm_length = 16
lstm_starts = window_starts(fit, lstm_length, max_dt=90.0, max_dist=2_000.0, require_normal=True)
lstm_starts = take_starts(lstm_starts, N_WINDOWS)
train_starts, calib_starts = split_starts(lstm_starts)
if len(calib_starts) < 10:
    calib_starts = train_starts[: max(10, len(train_starts) // 5)]
train_windows = np.clip(relative_xy_km(fit, train_starts, lstm_length), -30.0, 30.0)
calib_windows = relative_xy_km(fit, calib_starts, lstm_length)
lstm = LSTMAutoencoder()
train_torch(lstm, train_windows, lambda model, batch: ((model(batch) - batch) ** 2).mean(), N_EPOCHS)
calib_recon = batched_numpy(lstm, np.clip(calib_windows, -30.0, 30.0), lambda model, batch: model(batch))
calib_error = (calib_recon - calib_windows).reshape(len(calib_windows), -1)
error_mean = calib_error.mean(axis=0)
error_var = np.maximum(calib_error.var(axis=0), 1e-6)


def lstm_point_scores(pack):
    starts = window_starts(pack, lstm_length, max_dt=120.0, max_dist=np.inf, require_normal=False)
    if len(starts) == 0:
        return np.zeros(len(pack["lat"]))
    windows = relative_xy_km(pack, starts, lstm_length)
    recon = batched_numpy(lstm, np.clip(windows, -30.0, 30.0), lambda model, batch: model(batch))
    error = (recon - windows).reshape(len(starts), -1)
    window_score = diagonal_mahalanobis(error, error_mean, error_var)
    return scatter_max(len(pack["lat"]), starts, window_score, lstm_length)


fit_scores = lstm_point_scores(fit)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = lstm_point_scores(test)
print(f"  окон обучения {len(train_starts)}, порог {tau:.3g}, DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}")
show_result(
    "LSTM-автоэнкодер",
    test_scores > tau,
    time.perf_counter() - started,
    f"Окно {lstm_length}, скрытое состояние 8. Порог Махаланобиса {tau:.2f}. DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}.",
    test_scores,
)


# Temporal Convolutional Network

Каузальные свёртки по окну локальных координат предсказывают следующую точку. Остаток на штатных окнах описывается диагональным гауссовским законом, score — квадрат Махаланобиса этого остатка.


In [ ]:
class CausalTCN(nn.Module):
    """Каузальная свёрточная сеть: прогноз следующей точки по левому контексту."""

    def __init__(self, n_features=2, channels=16, levels=3, kernel=3):
        super().__init__()
        blocks = []
        pads = []
        in_channels = n_features
        dilation = 1
        for _ in range(levels):
            blocks.append(nn.Conv1d(in_channels, channels, kernel, dilation=dilation))
            pads.append((kernel - 1) * dilation)
            in_channels = channels
            dilation *= 2
        self.blocks = nn.ModuleList(blocks)
        self.pads = pads
        self.head = nn.Linear(channels, n_features)

    def forward(self, sequence):
        hidden = sequence.transpose(1, 2)
        for conv, pad in zip(self.blocks, self.pads):
            hidden = nn.functional.pad(hidden, (pad, 0))
            hidden = torch.relu(conv(hidden))
        return self.head(hidden[:, :, -1])


def tcn_pairs(pack, length, require_normal, limit=None):
    max_dist = 2_000.0 if require_normal else np.inf
    starts = window_starts(pack, length + 1, max_dt=90.0, max_dist=max_dist, require_normal=require_normal)
    if limit is not None:
        starts = take_starts(starts, limit)
    history = relative_xy_km(pack, starts, length)
    origin = pack["xy"][starts]
    target = ((pack["xy"][starts + length] - origin) / 1000.0).astype(np.float32)
    return starts, history, target


print("TCN")
started = time.perf_counter()
tcn_length = 16
pair_starts, history, target = tcn_pairs(fit, tcn_length, require_normal=True, limit=N_WINDOWS)
order = rng.permutation(len(pair_starts))
cut = max(1, int(len(order) * 0.85))
train_idx, calib_idx = order[:cut], order[cut:]


def train_predictor(model, features, targets, epochs):
    """Учит модель, которая по окну предсказывает вектор той же размерности, что и target."""
    features = np.clip(np.ascontiguousarray(features), -30.0, 30.0).astype(np.float32)
    targets = np.clip(np.ascontiguousarray(targets), -30.0, 30.0).astype(np.float32)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(features), torch.from_numpy(targets)),
        batch_size=256,
        shuffle=True,
    )
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for epoch in range(epochs):
        total, seen = 0.0, 0
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad()
            loss = ((model(batch_x) - batch_y) ** 2).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total += float(loss.detach()) * len(batch_x)
            seen += len(batch_x)
        print(f"    эпоха {epoch + 1}/{epochs}, loss {total / max(seen, 1):.5f}")
    model.eval()
    return model


tcn = CausalTCN()
train_predictor(tcn, history[train_idx], target[train_idx], N_EPOCHS)
with torch.no_grad():
    calib_pred = batched_numpy(tcn, np.clip(history[calib_idx], -30.0, 30.0), lambda model, batch: model(batch))
residual = calib_pred - target[calib_idx]
residual_mean = residual.mean(axis=0)
residual_var = np.maximum(residual.var(axis=0), 1e-6)


def tcn_point_scores(pack):
    starts, hist, truth = tcn_pairs(pack, tcn_length, require_normal=False)
    if len(starts) == 0:
        return np.zeros(len(pack["lat"]))
    pred = batched_numpy(tcn, np.clip(hist, -30.0, 30.0), lambda model, batch: model(batch))
    score = diagonal_mahalanobis(pred - truth, residual_mean, residual_var)
    point_score = np.zeros(len(pack["lat"]))
    np.maximum.at(point_score, starts + tcn_length, score)
    return point_score


fit_scores = tcn_point_scores(fit)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = tcn_point_scores(test)
print(f"  окон обучения {len(train_idx)}, порог {tau:.3g}, DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}")
show_result(
    "TCN",
    test_scores > tau,
    time.perf_counter() - started,
    f"Каузальное окно {tcn_length}. Порог Махаланобиса остатка {tau:.2f}. DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}.",
    test_scores,
)


# DONUT

Вариационный автоэнкодер учится на плоских окнах локальной траектории штатного хода. Score — оценка минус ELBO: ошибка восстановления плюс KL до стандартного нормального априорного. Это перенос VAE-постановки на окна траектории.


In [ ]:
class WindowVAE(nn.Module):
    """Полносвязный VAE для одного окна, записанного одним вектором."""

    def __init__(self, dim, latent=8):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(dim, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.mu = nn.Linear(32, latent)
        self.logvar = nn.Linear(32, latent)
        self.decoder = nn.Sequential(
            nn.Linear(latent, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, dim),
        )

    def forward(self, flat, sample):
        hidden = self.encoder(flat)
        mu = self.mu(hidden)
        logvar = self.logvar(hidden).clamp(-8.0, 8.0)
        if sample:
            latent = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        else:
            latent = mu
        return self.decoder(latent), mu, logvar


def vae_loss(model, batch):
    recon, mu, logvar = model(batch, sample=True)
    mse = ((recon - batch) ** 2).mean()
    kl = -0.5 * torch.mean(1.0 + logvar - mu.pow(2) - logvar.exp())
    return mse + kl


def vae_window_table(pack, starts, length, mean, std):
    windows = relative_xy_km(pack, starts, length).reshape(len(starts), -1)
    scaled = (windows - mean) / std
    return np.clip(scaled, -30.0, 30.0).astype(np.float32)


print("DONUT")
started = time.perf_counter()
vae_length = 16
vae_starts = take_starts(window_starts(fit, vae_length, max_dt=90.0, max_dist=2_000.0, require_normal=True), N_WINDOWS)
raw_windows = relative_xy_km(fit, vae_starts, vae_length).reshape(len(vae_starts), -1)
vae_mean = raw_windows.mean(axis=0)
vae_std = np.where(raw_windows.std(axis=0) < 1e-6, 1.0, raw_windows.std(axis=0))
vae_train = np.clip((raw_windows - vae_mean) / vae_std, -8.0, 8.0).astype(np.float32)
vae = WindowVAE(vae_train.shape[1])
train_torch(vae, vae_train, vae_loss, N_EPOCHS)


def donut_point_scores(pack):
    starts = window_starts(pack, vae_length, max_dt=120.0, max_dist=np.inf, require_normal=False)
    if len(starts) == 0:
        return np.zeros(len(pack["lat"]))
    table = vae_window_table(pack, starts, vae_length, vae_mean, vae_std)

    def score_batch(model, batch):
        recon, mu, logvar = model(batch, sample=False)
        mse = ((recon - batch) ** 2).sum(dim=-1)
        kl = -0.5 * torch.sum(1.0 + logvar - mu.pow(2) - logvar.exp(), dim=-1)
        return mse + kl

    window_score = batched_numpy(vae, table, score_batch)
    return scatter_max(len(pack["lat"]), starts, window_score, vae_length)


fit_scores = donut_point_scores(fit)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = donut_point_scores(test)
print(f"  порог {tau:.3g}, DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}")
show_result(
    "DONUT",
    test_scores > tau,
    time.perf_counter() - started,
    f"VAE, окно {vae_length}, латентность 8. Порог −ELBO {tau:.2f}. DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}.",
    test_scores,
)


# Anomaly Transformer

В каждом слое внимание сравнивается с гауссовским prior по расстоянию индексов внутри окна. Итоговый score позиции — произведение softmax от минус association discrepancy и квадрата ошибки восстановления. На вход подаются шаг в километрах, скорость и поворот курса.


In [ ]:
def symmetric_kl(attention, prior):
    """Симметричный KL по последней оси. prior имеет форму (L, L) и растягивается на батч."""
    prior = prior.unsqueeze(0)
    log_gap_left = torch.log(attention.clamp_min(1e-8)) - torch.log(prior.clamp_min(1e-8))
    log_gap_right = torch.log(prior.clamp_min(1e-8)) - torch.log(attention.clamp_min(1e-8))
    return 0.5 * ((attention * log_gap_left).sum(-1) + (prior * log_gap_right).sum(-1))


class AnomalyTransformer(nn.Module):
    """Уменьшенный Anomaly Transformer: внимание, prior-ассоциация и ошибка восстановления."""

    def __init__(self, n_features=4, d_model=32, n_heads=4, n_layers=2):
        super().__init__()
        self.input = nn.Linear(n_features, d_model)
        self.attention = nn.ModuleList(
            [nn.MultiheadAttention(d_model, n_heads, batch_first=True) for _ in range(n_layers)]
        )
        self.norm_attn = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.feed = nn.ModuleList(
            [
                nn.Sequential(nn.Linear(d_model, d_model * 2), nn.ReLU(), nn.Linear(d_model * 2, d_model))
                for _ in range(n_layers)
            ]
        )
        self.norm_feed = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.output = nn.Linear(d_model, n_features)
        self.raw_sigma = nn.Parameter(torch.zeros(n_layers))

    def forward(self, sequence):
        hidden = self.input(sequence)
        length = sequence.shape[1]
        index = torch.arange(length, device=sequence.device)
        distance2 = (index[:, None] - index[None, :]).float().pow(2)
        discrepancy = []
        for layer, attn in enumerate(self.attention):
            updated, weights = attn(hidden, hidden, hidden, need_weights=True, average_attn_weights=True)
            hidden = self.norm_attn[layer](hidden + updated)
            hidden = self.norm_feed[layer](hidden + self.feed[layer](hidden))
            sigma = torch.nn.functional.softplus(self.raw_sigma[layer]) + 1.0
            prior = torch.softmax(-distance2 / (2.0 * sigma**2), dim=-1)
            discrepancy.append(symmetric_kl(weights, prior))
        association = torch.stack(discrepancy, dim=0).mean(dim=0)
        recon = self.output(hidden)
        point_error = (recon - sequence).pow(2).sum(dim=-1)
        score = torch.softmax(-association, dim=-1) * point_error
        return recon, score


def transformer_windows(pack, starts, length, mean, std):
    index = starts[:, None] + np.arange(length)[None, :]
    step = np.stack(
        [pack["dx"][index] / 1000.0, pack["dy"][index] / 1000.0, pack["speed"][index], pack["dpsi"][index]],
        axis=-1,
    )
    scaled = (step - mean) / std
    return np.clip(scaled, -30.0, 30.0).astype(np.float32)


print("Anomaly Transformer")
started = time.perf_counter()
at_length = 32
at_starts = take_starts(window_starts(fit, at_length, max_dt=90.0, max_dist=2_000.0, require_normal=True), N_WINDOWS)
at_index = at_starts[:, None] + np.arange(at_length)[None, :]
at_raw = np.stack(
    [
        fit["dx"][at_index] / 1000.0,
        fit["dy"][at_index] / 1000.0,
        fit["speed"][at_index],
        fit["dpsi"][at_index],
    ],
    axis=-1,
)
at_mean = at_raw.mean(axis=(0, 1))
at_std = at_raw.std(axis=(0, 1))
at_std = np.where(at_std < 1e-6, 1.0, at_std)
at_train = np.clip((at_raw - at_mean) / at_std, -8.0, 8.0).astype(np.float32)
transformer = AnomalyTransformer()


def transformer_loss(model, batch):
    recon, _ = model(batch)
    return ((recon - batch) ** 2).mean()


train_torch(transformer, at_train, transformer_loss, N_EPOCHS)


def transformer_point_scores(pack):
    starts = window_starts(pack, at_length, max_dt=120.0, max_dist=np.inf, require_normal=False)
    starts = starts[::2]
    if len(starts) == 0:
        return np.zeros(len(pack["lat"]))
    windows = transformer_windows(pack, starts, at_length, at_mean, at_std)
    scores = batched_numpy(transformer, windows, lambda model, batch: model(batch)[1])
    return scatter_max(len(pack["lat"]), starts, scores, at_length)


fit_scores = transformer_point_scores(fit)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = transformer_point_scores(test)
print(f"  порог {tau:.3g}, DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}")
show_result(
    "Anomaly Transformer",
    test_scores > tau,
    time.perf_counter() - started,
    f"Окно {at_length}, 2 слоя, d_model 32. Порог {tau:.3g}. DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}.",
    test_scores,
)


# STGVAD

В окне траектории строится граф: сообщение идёт только от более ранних точек, а пространственное ребро добавляется, если точки ближе 5 км. Сеть предсказывает положение точки по соседям из прошлого. Score — ошибка этого прогноза в метрах. Идентификатора судна в файле нет, поэтому пространственные связи ищутся внутри окна одной последовательности.


In [ ]:
class STGNN(nn.Module):
    """Два слоя message passing. Собственные координаты точки в расчёт не входят."""

    def __init__(self):
        super().__init__()
        self.own1 = nn.Linear(2, 32)
        self.msg1 = nn.Linear(4, 32)
        self.own2 = nn.Linear(32, 32)
        self.msg2 = nn.Linear(32, 32)
        self.head = nn.Linear(32, 2)

    def forward(self, own, neigh, adj):
        hidden = torch.relu(self.own1(own) + self.msg1(torch.matmul(adj, neigh)))
        hidden = torch.relu(self.own2(hidden) + self.msg2(torch.matmul(adj, hidden)))
        return self.head(hidden)


def graph_batch(pack, starts, length):
    index = starts[:, None] + np.arange(length)[None, :]
    xy = pack["xy"][index]
    relative = xy - xy[:, :1, :]
    distance = np.linalg.norm(relative[:, :, None, :] - relative[:, None, :, :], axis=-1)
    past = np.tri(length, k=-1, dtype=np.float32)
    adj = ((distance < 5_000.0) & (distance > 0.0)).astype(np.float32) * past
    adj /= np.maximum(adj.sum(axis=-1, keepdims=True), 1.0)
    speed = (pack["speed"][index] / 10.0).astype(np.float32)
    turn = pack["dpsi"][index].astype(np.float32)
    own = np.clip(np.stack([speed, turn], axis=-1), -30.0, 30.0).astype(np.float32)
    neigh = np.clip(
        np.stack([relative[:, :, 0] / 1000.0, relative[:, :, 1] / 1000.0, speed, turn], axis=-1),
        -30.0,
        30.0,
    ).astype(np.float32)
    target = (relative / 1000.0).astype(np.float32)
    return own, neigh, adj.astype(np.float32), target


def train_stgnn(model, own, neigh, adj, target, epochs):
    loader = DataLoader(
        TensorDataset(
            torch.from_numpy(own),
            torch.from_numpy(neigh),
            torch.from_numpy(adj),
            torch.from_numpy(np.clip(target, -30.0, 30.0)),
        ),
        batch_size=128,
        shuffle=True,
    )
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for epoch in range(epochs):
        total, seen = 0.0, 0
        for batch_own, batch_neigh, batch_adj, batch_y in loader:
            batch_own = batch_own.to(DEVICE)
            batch_neigh = batch_neigh.to(DEVICE)
            batch_adj = batch_adj.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad()
            pred = model(batch_own, batch_neigh, batch_adj)
            loss = ((pred[:, 1:, :] - batch_y[:, 1:, :]) ** 2).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total += float(loss.detach()) * len(batch_own)
            seen += len(batch_own)
        print(f"    эпоха {epoch + 1}/{epochs}, loss {total / max(seen, 1):.5f}")
    model.eval()
    return model


def stgnn_point_scores(model, pack, length):
    starts = window_starts(pack, length, max_dt=120.0, max_dist=np.inf, require_normal=False)
    starts = starts[::2]
    if len(starts) == 0:
        return np.zeros(len(pack["lat"]))
    pieces = []
    batch = 512
    for cursor in range(0, len(starts), batch):
        own, neigh, adj, target = graph_batch(pack, starts[cursor : cursor + batch], length)
        with torch.no_grad():
            pred = (
                model(
                    torch.from_numpy(own).to(DEVICE),
                    torch.from_numpy(neigh).to(DEVICE),
                    torch.from_numpy(adj).to(DEVICE),
                )
                .cpu()
                .numpy()
            )
        error = np.linalg.norm(pred - target, axis=-1) * 1000.0
        error[:, 0] = 0.0
        pieces.append(error)
    return scatter_max(len(pack["lat"]), starts, np.concatenate(pieces, axis=0), length)


print("STGVAD")
started = time.perf_counter()
graph_length = 16
graph_starts = take_starts(
    window_starts(fit, graph_length, max_dt=90.0, max_dist=2_000.0, require_normal=True),
    N_WINDOWS,
)
own_fit, neigh_fit, adj_fit, target_fit = graph_batch(fit, graph_starts, graph_length)
stgnn = train_stgnn(STGNN(), own_fit, neigh_fit, adj_fit, target_fit, N_EPOCHS)
fit_scores = stgnn_point_scores(stgnn, fit, graph_length)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = stgnn_point_scores(stgnn, test, graph_length)
print(f"  порог {tau:.3g} м, DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}")
show_result(
    "STGVAD",
    test_scores > tau,
    time.perf_counter() - started,
    f"Окно {graph_length}, пространственный радиус 5 км. Порог {tau:.1f} м. DL обучения {LAST_TRAIN_DL:.4g} м, F1 {train_f1:.3f}.",
    test_scores,
)


# Максимальная физически согласованная траектория

Точка остаётся штатной, если от последней принятой точки до неё скорость не выше `v_max` и форма окна не выше порога. Иначе она аномалия, а якорь не сдвигается: дальнее облако, в которое нельзя попасть с допустимой скоростью, не становится новым «штатным» участком. Пара (`v_max`, порог формы) выбирается по MCC, в полосе `MCC_BAND` — по меньшему distance loss.

Непрерывный score для PR-AUC — превышение скорости над `v_max`, превышение формы окна и смещение от последней принятой точки в километрах. У принятой точки score равен 0.


In [ ]:
def mpct_detect(pack, v_max, shape_limit):
    """Аномалия, если точка недостижима от последней принятой или окно слишком извилистое.

    Якорь — последняя точка, прошедшая оба порога. Отклонённая точка якорь не заменяет.
    """
    lat = pack["lat"]
    lon = pack["lon"]
    t_sec = pack["t"]
    shape = pack["shape"]
    count = len(lat)
    predicted = np.zeros(count, dtype=bool)
    scores = np.zeros(count, dtype=float)
    last = 0
    for index in range(1, count):
        distance = float(haversine_m(lat[last], lon[last], lat[index], lon[index]))
        delta = max(float(t_sec[index] - t_sec[last]), 1.0)
        speed = distance / delta
        shape_excess = max(0.0, float(shape[index]) - shape_limit)
        if speed > v_max or shape[index] > shape_limit:
            predicted[index] = True
            scores[index] = max(0.0, speed - v_max) + shape_excess + distance / 1000.0
        else:
            last = index
    return predicted, scores


print("MPCT")
started = time.perf_counter()
records = []
for speed_limit in (8.0, 10.0, 12.0, 15.0, 20.0):
    for shape_limit in (1.5, 2.0, 3.0):
        predicted, _ = mpct_detect(fit, speed_limit, shape_limit)
        record = operating_record(fit, predicted, speed_limit)
        record["speed"] = float(speed_limit)
        record["shape_limit"] = float(shape_limit)
        print(
            f"  v_max {speed_limit:.0f} м/с, форма {shape_limit:.1f}: "
            f"доля исключённых {predicted.mean():.1%}, "
            f"MCC {record['mcc']:.3f}, DL {record['loss']:.4g} м, F1 {record['f1']:.3f}"
        )
        records.append(record)
chosen, best_mcc, band_size = pick_operating_point(records)
best_speed = chosen["speed"]
best_shape = chosen["shape_limit"]
best_f1 = chosen["f1"]
print(
    f"  выбрано v_max {best_speed:.0f} м/с, форма {best_shape:.1f}: "
    f"MCC {chosen['mcc']:.3f}, лучший MCC {best_mcc:.3f}, "
    f"в полосе {band_size}, DL {chosen['loss']:.4g} м, F1 {best_f1:.3f}"
)
test_pred, test_scores = mpct_detect(test, best_speed, best_shape)
print(
    f"  медиана score: вне маршрута {np.median(test_scores[test_pred]) if np.any(test_pred) else 0:.4g}, "
    f"на маршруте {np.median(test_scores[~test_pred]) if np.any(~test_pred) else 0:.4g}"
)
remember_train_operating_point(chosen["loss"], best_f1, chosen["mcc"])
show_result(
    "MPCT",
    test_pred,
    time.perf_counter() - started,
    (
        f"Якорь — последняя принятая точка. "
        f"v_max {best_speed:.0f} м/с, форма окна ≤ {best_shape:.1f}. "
        f"DL обучения {LAST_TRAIN_DL:.4g} м, F1 {best_f1:.3f}."
    ),
    test_scores,
)




# Сводка

Таблица собрана из `CalculateMetrics.calculate_all_metrics` на тестовой выборке. Модели упорядочены по MCC: он растёт, когда одновременно растут и доля пойманных аномалий, и доля сохранённого штатного хода. При близком MCC выше строка с меньшим distance loss. Постоянные разметки в этот порядок не входят: они показывают края шкалы.

Положительный класс point-wise и affiliation метрик — аномалия, метка `0`. PR-AUC считается по непрерывному score: у forecasting- и reconstruction-методов это ошибка модели, у DBSCAN — max(k-distance, расстояние до core), у MPCT — превышение скорости и формы окна плюс смещение от последней принятой точки. У постоянных разметок score нет, поэтому PR-AUC не определён. Тепловая карта показывает только метрики в диапазоне от 0 до 1. Столбцы «DL обуч.», «F1 обуч.» и «MCC обуч.» — качество выбранного порога на всей обучающей выборке.



In [ ]:
summary = pd.DataFrame(RESULTS)
if summary.empty:
    raise RuntimeError("Нет результатов: сводка собирается после ячеек моделей.")
if "mcc" not in summary.columns:
    raise RuntimeError("В результатах нет MCC: заново выполните ячейки моделей.")

ref_m = reference_route_length(test)
baselines = ["Всегда штатная", "Всегда аномалия"]
is_baseline = summary["Метод"].isin(baselines)
models = summary.loc[~is_baseline].copy()


def finite_column(frame, key, missing):
    values = frame[key].to_numpy(dtype=float)
    return np.where(np.isfinite(values), values, missing)


models["_mcc"] = -finite_column(models, "mcc", -1.0)
models["_dl"] = finite_column(models, "distance_loss", 1e18)
models["_f1"] = -finite_column(models, "point_f1", -1.0)
models = models.sort_values(["_mcc", "_dl", "_f1", "Метод"], kind="mergesort")
baseline_rows = summary.loc[is_baseline].copy()
ordered = pd.concat([models, baseline_rows], ignore_index=True)

table_columns = [
    "Метод",
    "mcc",
    "distance_loss",
    "distance_loss_ratio",
    "point_f1",
    "point_precision",
    "point_recall",
    "pr_auc",
    "affiliation_f1",
    "geodesic_rmse",
    "hausdorff_distance",
    "detection_delay_median",
    "DL обучения, м",
    "F1 обучения",
    "MCC обучения",
    "Время, с",
]
printable = ordered[table_columns].copy()
headers = {
    "mcc": "MCC",
    "distance_loss": "DL, м",
    "distance_loss_ratio": "DLR",
    "point_f1": "Point F1",
    "point_precision": "Precision",
    "point_recall": "Recall",
    "pr_auc": "PR-AUC",
    "affiliation_f1": "Affiliation F1",
    "geodesic_rmse": "RMSE, м",
    "hausdorff_distance": "Hausdorff, м",
    "detection_delay_median": "Delay, с",
    "DL обучения, м": "DL обуч., м",
    "F1 обучения": "F1 обуч.",
    "MCC обучения": "MCC обуч.",
    "Время, с": "Время, с",
}
printable = printable.rename(columns=headers)
print(f"Эталон маршрута на тесте: {ref_m:,.2f} м.")
print("Порядок строк: сначала MCC. При близком MCC — меньший distance loss, затем Point F1.")
print(
    printable.to_string(
        index=False,
        float_format=lambda value: "—" if not np.isfinite(value) else f"{value:.4f}",
    )
)
if ref_m == 0.0:
    print(
        "Эталон на тесте нулевой: в тестовом отрезке нет точек move. "
        "Distance loss равен длине пути, который модель оставила классом move."
    )

rate_columns = [key for key, _ in RATE_FIELDS]
heat = np.ma.masked_invalid(ordered[rate_columns].to_numpy(dtype=float, copy=True))
fig, ax = plt.subplots(figsize=(12.4, 0.42 * len(ordered) + 1.8))
image = ax.imshow(heat, cmap="YlOrRd", vmin=0.0, vmax=1.0, aspect="auto")
ax.set_xticks(range(len(RATE_FIELDS)))
ax.set_xticklabels([label for _, label in RATE_FIELDS], rotation=30, ha="right")
ax.set_yticks(range(len(ordered)))
ax.set_yticklabels(ordered["Метод"])
for row in range(heat.shape[0]):
    for col in range(heat.shape[1]):
        value = heat[row, col]
        label = "—" if np.ma.is_masked(value) else f"{float(value):.3f}"
        ax.text(col, row, label, ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=ax, fraction=0.03, pad=0.02)
ax.set_title("Доли от 0 до 1. Строки упорядочены по MCC")
fig.tight_layout()
try:
    from IPython import get_ipython

    interactive = get_ipython() is not None
except Exception:
    interactive = False
if interactive:
    plt.show()
else:
    plt.close(fig)


def fmt_rate(value):
    return "—" if not np.isfinite(value) else f"{float(value):.4f}"


print()
print("Методы по MCC, без постоянных разметок:")
for _, item in models.iterrows():
    print(
        f"  {item['Метод']:<24} MCC {fmt_rate(item['mcc'])}   "
        f"DL {item['distance_loss']:.2f} м   "
        f"F1 {fmt_rate(item['point_f1'])}   "
        f"PR-AUC {fmt_rate(item['pr_auc'])}"
    )
leader = models.iloc[0]
print()
print(
    f"По разделению классов впереди {leader['Метод']}: "
    f"MCC {fmt_rate(leader['mcc'])}, DL {leader['distance_loss']:.2f} м, "
    f"Point F1 {fmt_rate(leader['point_f1'])}."
)



# Выводы по прогону

Фрагмент `[200000:350000)`, без точек `stand`: обучение 43 158 точек (62% `anomaly`), тест 100 000 (54% `anomaly`, 46% `move`). Эталон маршрута на тесте — 2 182 км. Сырая ломаная того же участка длиннее примерно в 97 раз: если ничего не вырезать, distance loss равен 210 млн м (DLR 96). На тесте 19 аномальных событий.

Две шкалы здесь расходятся, и это главный результат прогона. MCC и point-wise F1 отвечают, узнана ли точка. Distance loss отвечает, какую ломаную модель оставила как маршрут. Разметка «все точки аномальны» даёт DL 2,18 млн м (DLR 1) и при этом MCC 0: маршрут пуст, штатный ход уничтожен. Поэтому порог и сводка идут от MCC, а длина сравнивается уже среди моделей, которые реально различают классы.

## Кто разделяет классы

По MCC на тесте впереди три модели.

DBSCAN: MCC 0,82, F1 0,92, полнота 0,98, точность 0,87, PR-AUC 0,96. Сохранено 83% точек `move`. Все 19 событий найдены, средняя задержка 6 с. Это лучшая точечная разметка в прогоне. Оговорка: кластеры строятся по тестовому облаку, а на обучении у выбранного `eps` MCC был только 0,42. Цифра на тесте частично описывает тот же образец, на котором модель только что выучила плотность.

Robust PCA: MCC 0,74, F1 0,88, полнота 0,85, точность 0,90, сохранено 89% `move`. По точкам это второй результат. По длине маршрута он проваливается: DL 175 млн м, DLR 80, Hausdorff 13 058 км — та же верхняя точка, что у разметки «ничего не вырезать». Несколько пропущенных скачков держат всю лишнюю длину, хотя большинство точек помечено верно.

MPCT (`v_max` 8 м/с, форма окна ≤ 1,5): MCC 0,74, F1 0,87, точность 0,91, полнота 0,84, сохранено 90% `move`. Найдено 18 из 19 событий, средняя задержка 54 с. По точкам он рядом с Robust PCA. По длине он один в другом порядке: DL 2,95 млн м, DLR 1,35. Ошибка сравнима с эталоном, а не в десятки раз больше него.

## Длина маршрута у остальных

Дальше по разумной ошибке длины, уже с заметным MCC:

- Isolation Forest: MCC 0,66, полнота 0,97, но сохранено только 65% `move`. DL 11,2 млн м (DLR 5,1). Зато самый маленький геометрический хвост: RMSE 26 км, Hausdorff 81 км. Лес вырезает дальние выбросы и вместе с ними треть штатного хода. Порог −0,17 стоит внутри облака штатных score (медиана `move` −0,18).
- Фильтр Калмана: MCC 0,65, точность 0,92, полнота 0,71, сохранено 93% `move`, DL 16,2 млн м (DLR 7,4), RMSE 46 км. Score разделён хорошо (медиана `move` 0,0003 против 19 у аномалий), но выбранный порог 0,007 намного ниже теоретического χ²₀.₉₉₉ = 13,8. Ворота по MCC пропускают больше слабых отклонений, чем жёсткий физический порог.
- ARIMA: самая высокая точность среди рабочих моделей, 0,94, и сохранено 96% `move`, но полнота только 0,53 и DL 58 млн м. События она видит (affiliation F1 0,93, 18 из 19), а точки внутри эпизода пропускает. Коэффициенты сняты с гладкого куска в 2 167 точек.
- HMM и фильтр частиц держат штатный ход (88% и 95% `move`), но полнота 0,67 и 0,35. Длина остаётся в 32 и 41 раз больше эталона.

## Где перенос на тест ломается

Нейросети на обучении выглядят сильно: MCC обучения 0,69–0,83, доля сохранённого `move` 89–98%. На тесте это схлопывается. TCN: MCC 0,06 при F1 обучения 0,84. LSTM и DONUT: MCC 0,24. STGVAD: 0,29. Anomaly Transformer держится лучше остальных, MCC 0,50, но сохраняет только 57% `move`. У LSTM, TCN, DONUT, STGVAD и Anomaly Transformer Hausdorff один и тот же, 13 058 км: хотя бы один дальний скачок остаётся в принятом маршруте, и длина остаётся в 50–80 раз больше эталона. Шесть эпох на окнах обучающего хода не переносят порог на тест, где аномалии чаще выглядят как медленное дрожание, а не как те скачки, на которых сеть выучила большую ошибку восстановления.

One-Class SVM на обучении даёт MCC 0,84, на тесте 0,53 и сохраняет 61% `move`. Граница, снятая с 4 000 штатных точек обучения, на тесте слишком тесная.

Hampel и LOF классы почти не разделяют. Медианы score: Hampel 0,52 у `move` и 0,67 у аномалий, LOF 1,05 и 1,03, причём у LOF аномалии даже плотнее штатного хода. Полнота 0,16 и 0,11, F1 0,26 и 0,19, DL около 50 эталонов. Локальный выброс по скорости или плотности не видит длинный участок, который шагает так же ровно, как `move`, но стоит не на маршруте.

Гауссовский процесс в этой форме не работает. Медиана score у `move` выше, чем у аномалий: 1,86 против 0,13. Порог падает до 0,002, на тесте аномалией помечено 99% точек, штатный ход не сохранён, MCC −0,11. Прогноз шага с длиной корреляции 6 часов страннее для ровного хода, чем для меток, которые в файле названы аномалией.

## Что следует из этого

Для длины маршрута опорная модель — MPCT. Он один удерживает и оба класса (около 90% штатного хода и 84% аномалий), и ошибку длины в полтора эталона, а не в десятки раз. Остаток в 3 000 км — это точки, которые от последней принятой ещё достижимы на 8 м/с и при этом стоят в файле как аномалия. Порог скорости их не видит.

Для точечной разметки опорный DBSCAN: лучший MCC и F1, почти все аномалии найдены. Плата — DLR 9 и то, что модель подгоняется по тесту, а не только по обучению. Isolation Forest и Калман — запасной вариант, если нужен прогноз без перестроения на тесте: они хуже DBSCAN по точкам и лучше нейросетей по длине, но до MPCT им далеко.

Гауссовский процесс, TCN, Hampel и LOF в этом прогоне к решению не годятся: первый переворачивает классы, второй не переносит порог, последние два не отделяют ровный чужой участок от ровного хода.
